# Overnight run (commit mode): unseen-item transfer, model scale, breadth seed 1
Settings: Accelerator **GPU T4 x2**, Internet **On**, Add Input -> the private dataset holding `kaggle_pkg3/` (only that one).
Run with **Save Version -> Save & Run All (Commit)**. Runs on Kaggle's servers; your computer may sleep. Output: /kaggle/working (zips per stage + status.json).

In [ ]:
# 1. install + GPU check
!pip install -q -U bitsandbytes peft accelerate "transformers>=4.45"
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2. setup: locate package, build repo, resolve paths
import os, glob, shutil, subprocess, sys, time, json, traceback
os.environ["HF_HOME"] = "/tmp/hf"; os.makedirs("/tmp/hf", exist_ok=True)
def find(name, must_contain=None):
    for h in sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)):
        if must_contain is None or must_contain in open(h, errors="ignore").read():
            return h
    raise AssertionError(f"{name} not found (must_contain={must_contain}). Attach the dataset. Seen: {os.listdir('/kaggle/input')}")
INFER = find("kaggle_pew_infer.py", "--item-range")
TRAIN = find("kaggle_pew_train.py"); P0 = find("kaggle_p0.py"); SCALE = find("kaggle_zs_scale.py")
LABAD = os.path.dirname(find("adapter_config.json"))
src = os.path.dirname(os.path.dirname(find("config.py", "PEW_RAW_CSV_NAME")))
REPO = "/tmp/repo"
if not os.path.exists(f"{REPO}/src/config.py"):
    shutil.copytree(src, REPO)
for f in ("data/processed/pew_india_2021.parquet", "data/processed/pew_folds.json", "data/processed/pew_selected_items.json", "data/reference/pew_codebook.json"):
    assert os.path.exists(f"{REPO}/{f}"), f"missing {f}"
shutil.copy(find("kaggle_common.py"), os.path.dirname(TRAIN)) if not os.path.exists(os.path.join(os.path.dirname(TRAIN), "kaggle_common.py")) else None
print("INFER", INFER, "\nTRAIN", TRAIN, "\nSCALE", SCALE, "\nLAB adapter", LABAD, "\nREPO", REPO)

In [ ]:
# 3. the overnight pipeline. Every stage is wrapped: a failure is logged and the next stage still runs.
from huggingface_hub import snapshot_download
T0 = time.time(); W = "/kaggle/working"; STATUS = {}
def hrs(): return (time.time() - T0) / 3600
def save_status(**kw):
    STATUS.update(kw); json.dump(STATUS, open(f"{W}/status.json", "w"), indent=1)
def launch(cmd, gpus, log):
    return subprocess.Popen([sys.executable] + cmd, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpus}, stdout=open(log, "a"), stderr=subprocess.STDOUT)
def wait(procs, logs, every=120):
    while any(p.poll() is None for p in procs):
        time.sleep(every)
        for lg in logs:
            try: print(os.path.basename(lg), "|", open(lg).read().splitlines()[-1][:180], f"| {hrs():.2f} h", flush=True)
            except Exception: pass
    return [p.returncode for p in procs]
def zipdir(name, d):
    if os.path.isdir(d):
        shutil.make_archive(f"{W}/{name}", "zip", d); print("zipped", name, round(os.path.getsize(f"{W}/{name}.zip")/1e6, 1), "MB", flush=True)
def stage(name, fn, max_start_h=10.0):
    if hrs() > max_start_h:
        print(f"[skip] {name}: {hrs():.1f} h elapsed"); save_status(**{name: "skipped_time"}); return
    print(f"\n===== {name} @ {hrs():.2f} h =====", flush=True)
    try:
        fn(); save_status(**{name: f"ok @ {hrs():.2f} h"})
    except Exception as e:
        traceback.print_exc(); save_status(**{name: f"FAILED: {str(e)[:200]} @ {hrs():.2f} h"})

def s_unseen():  # lab adapter + base on 32 unseen items, 1000 respondents, both GPUs sharded
    out = f"{W}/out_unseen"; os.makedirs(out, exist_ok=True)
    ps = []
    for s in (0, 1):
        ps.append(launch([INFER, "--repo", REPO, "--adapter", LABAD, "--out-dir", out, "--item-range", "8:40", "--n-resp", "1000",
                          "--shard", str(s), "--nshards", "2", "--batch", "16", "--max-hours", "3.5", "--conditions", "adapter_raw,base_raw"],
                         str(s), f"{W}/log_unseen_s{s}.txt")); time.sleep(120)
    rc = wait(ps, [f"{W}/log_unseen_s0.txt", f"{W}/log_unseen_s1.txt"]); print("rc", rc)
    zipdir("result_unseen", out)

def s_scale(model, n_resp, batch, device_map, gpus, max_h, shards):
    def fn():
        snapshot_download(model, allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json"])
        out = f"{W}/out_scale"; os.makedirs(out, exist_ok=True); ps, logs = [], []
        for s in range(shards):
            lg = f"{W}/log_scale_{model.split('/')[-1]}_s{s}.txt"; logs.append(lg)
            ps.append(launch([SCALE, "--repo", REPO, "--model", model, "--out-dir", out, "--item-range", "0:8", "--n-resp", str(n_resp),
                              "--shard", str(s), "--nshards", str(shards), "--batch", str(batch), "--device-map", device_map,
                              "--max-hours", str(max_h), "--conditions", "base_raw,base_chat"], gpus[s], lg)); time.sleep(90)
        print("rc", wait(ps, logs)); zipdir("result_scale", out)
        shutil.rmtree(f"/tmp/hf/hub/models--{model.replace('/', '--')}", ignore_errors=True)  # free disk
    return fn

def s_seed(seed, n_resp=2000, max_hours=4.5):
    def fn():
        snapshot_download("Qwen/Qwen2.5-7B-Instruct", allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json"])
        D = f"{W}/seed{seed}"; os.makedirs(D, exist_ok=True)
        common = ["--repo", REPO, "--n-resp", str(n_resp), "--seed", str(seed), "--max-hours", str(max_hours)]
        pB = launch([TRAIN, *common, "--out-dir", f"{D}/train_B", "--item-range", "0:8"], "0", f"{D}/log_train_B.txt"); time.sleep(120)
        pA = launch([TRAIN, *common, "--out-dir", f"{D}/train_A", "--item-range", "0:24"], "1", f"{D}/log_train_A.txt")
        print("train rc", wait([pB, pA], [f"{D}/log_train_B.txt", f"{D}/log_train_A.txt"]))
        zipdir(f"seed{seed}_train_logs", D)
        adB, adA = f"{D}/train_B/adapter_final", f"{D}/train_A/adapter_final"
        assert os.path.exists(adB) and os.path.exists(adA), "training did not finish"
        ev = ["--repo", REPO, "--item-range", "0:40", "--n-resp", "500", "--nshards", "1", "--shard", "0", "--batch", "16", "--max-hours", "2.5"]
        e1 = launch([INFER, *ev, "--adapter", adB, "--out-dir", f"{D}/eval_B", "--conditions", "adapter_raw"], "0", f"{D}/log_eval_B.txt"); time.sleep(120)
        e2 = launch([INFER, *ev, "--adapter", adA, "--out-dir", f"{D}/eval_A", "--conditions", "adapter_raw"], "1", f"{D}/log_eval_A.txt")
        print("eval rc", wait([e1, e2], [f"{D}/log_eval_B.txt", f"{D}/log_eval_A.txt"]))
        p = launch([P0, "--repo", REPO, "--adapters", f"lab={LABAD},A={adA},B={adB}", "--out", f"{D}/p0.json", "--item-range", "0:40"], "0", f"{D}/log_p0.txt")
        print("p0 rc", wait([p], [f"{D}/log_p0.txt"], every=30))
        for ad in ("train_A", "train_B"):   # keep result zips small: drop optimizer checkpoints
            shutil.rmtree(f"{D}/{ad}/ckpt", ignore_errors=True)
        zipdir(f"seed{seed}_results", D)
    return fn

save_status(started=time.strftime("%Y-%m-%d %H:%M:%S"))
stage("1_unseen_transfer", s_unseen)
stage("2_scale_14B", s_scale("Qwen/Qwen2.5-14B-Instruct", 500, 8, "0", ["0", "1"], 1.5, 2))
stage("3_breadth_seed1", s_seed(1), max_start_h=3.5)
stage("4_scale_32B", s_scale("Qwen/Qwen2.5-32B-Instruct", 300, 4, "auto", ["0,1"], 2.5, 1), max_start_h=8.0)
save_status(finished=time.strftime("%Y-%m-%d %H:%M:%S"), total_hours=round(hrs(), 2))
print(json.dumps(STATUS, indent=1))